loading and inspecting the data

In [1]:
import pandas as pd

df = pd.read_csv("data.csv")  # update filename

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


checking the structure

In [2]:
print(df.shape)
print(df.columns)
print(df['sentiment'].value_counts())

(50000, 2)
Index(['review', 'sentiment'], dtype='object')
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


preprocessing

In [4]:
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ROCKY\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


main function (sentimental analysis)

In [5]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove punctuation
    words = text.split()
    
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]
    
    return " ".join(words)

applying the main function

In [7]:
from tqdm import tqdm
tqdm.pandas()

df['clean_text'] = df['review'].progress_apply(preprocess_text)

100%|███████████████████████████████████████████████████████████████████████████| 50000/50000 [03:28<00:00, 239.85it/s]


In [9]:
y = df['sentiment']

 4. Labels

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['clean_text'])

train test split

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train models

In [13]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

LogisticRegression()

In [14]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

MultinomialNB()

In [19]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=50, max_depth=10)
rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=50)

evaluation function:

In [20]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate_model(model, name):
    y_pred = model.predict(X_test)
    
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

In [21]:
evaluate_model(lr, "Logistic Regression")
evaluate_model(nb, "Naive Bayes")
evaluate_model(rf, "Decision Tree")


Logistic Regression
Accuracy: 0.8823
              precision    recall  f1-score   support

    negative       0.89      0.87      0.88      4961
    positive       0.87      0.90      0.88      5039

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


Naive Bayes
Accuracy: 0.848
              precision    recall  f1-score   support

    negative       0.86      0.83      0.84      4961
    positive       0.84      0.86      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000


Decision Tree
Accuracy: 0.8188
              precision    recall  f1-score   support

    negative       0.86      0.76      0.81      4961
    positive       0.79      0.88      0.83      5039

    accuracy                           0.82     10000
   macro avg       0.82      

camparisn table

In [24]:
results = {
    "Model": ["Logistic Regression", "Naive Bayes", "Random forest"],
    "Accuracy": [
        accuracy_score(y_test, lr.predict(X_test)),
        accuracy_score(y_test, nb.predict(X_test)),
        accuracy_score(y_test, rf.predict(X_test))
    ]
}

pd.DataFrame(results)

,Model,Accuracy
0,Logistic Regression,0.8823
1,Naive Bayes,0.8480
2,Random forest,0.8188
